In [8]:
import torch

from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, IterableDataset

import json
from pyarrow import parquet as pq

import numpy as np

In [9]:
class LoadLanguageData(IterableDataset):
    
    def __init__(self, path:str, columns:list) -> None:

        super().__init__()

        self.cols = columns
        self.path = path
        self.dataset = pq.ParquetDataset(self.path)

    def __iter__(self):

        for fragment in self.dataset.fragments:

            table = fragment.to_table()
            rows = table.to_pylist()

            for row in rows:

                yield {
                    'input_ids' : torch.tensor(row[self.cols[0]], dtype=torch.long),
                    'output_ids': torch.tensor(row[self.cols[1]], dtype=torch.long),
                    'target_ids': torch.tensor(row[self.cols[2]], dtype=torch.long)
                }

def wrapper_collate_batch(pad_value_in, pad_value_out):

    def collate_batch(batch):

        input_list  = [item['input_ids']  for item in batch]
        output_list = [item['output_ids'] for item in batch]
        target_list = [item['target_ids'] for item in batch]

        input_padded  = pad_sequence(input_list,  batch_first=True, padding_value=pad_value_in)
        output_padded = pad_sequence(output_list, batch_first=True, padding_value=pad_value_out)
        target_padded = pad_sequence(target_list, batch_first=True, padding_value=pad_value_out)

        return (input_padded, output_padded, target_padded)
    
    return collate_batch


def load_tokenizer(path:str) -> tuple:

    with open(path) as f:
        tokenizer = json.load(f)
    
    word2index = tokenizer["vocab"]
    index2word = {int(key):value for key, value in tokenizer["inverse_vocab"].items()}

    return word2index, index2word


def index2word(input:torch.tensor, mapper:dict) -> torch.tensor:

    max_key = max(mapper.keys())
    names_array = np.array(["Unknown"] * (max_key + 1), dtype=object)

    for idx, val in mapper.items():
        names_array[idx] = val

    mapped_output = names_array[input.numpy()]

    print(mapped_output)


In [10]:
word2index_eng, index2word_eng = load_tokenizer("../data/main/vocabs/eng_tokenizer.json")
word2index_spa, index2word_spa = load_tokenizer("../data/main/vocabs/spa_tokenizer.json")

pad_value_eng = word2index_eng["[PAD]"]
pad_value_spa = word2index_spa["[PAD]"]

In [11]:
train_path = "../data/main/spa-eng/train_spa_eng"
train_schema = pq.ParquetDataset(train_path)
train_cols = train_schema.schema.names

test_path = "../data/main/spa-eng/test_spa_eng"
test_schema = pq.ParquetDataset(test_path)
test_cols = test_schema.schema.names

In [12]:
train_data = LoadLanguageData(train_path, train_cols)
train_loader = DataLoader(train_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa))

test_data = LoadLanguageData(train_path, test_cols)
test_loader = DataLoader(test_data, batch_size=64, collate_fn=wrapper_collate_batch(pad_value_eng, pad_value_spa))

In [13]:
for eng_in, spa_out, spa_t in train_loader:
    break

print(eng_in.shape)
print(spa_out.shape)
print(spa_t.shape)

torch.Size([64, 14])
torch.Size([64, 11])
torch.Size([64, 11])
